# RainGuard – Module 7: Model Comparison, Interpretation & Final Evaluation

**Goal:** consolidate the outputs from Modules 5 (regression) and 6 (classification), compare the selected models on the same chronological holdout period, inspect prediction errors, interpret model features where supported, and export a final evaluation summary.

**Run order:** Complete and run all cells in `RainGuard_05_Regression.ipynb` and `RainGuard_06_Classification.ipynb` first. This notebook reads their saved CSV/JSON/model artifacts; it does not retrain the models.

## 1. Imports and project paths

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid")

cwd = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(cwd, "..")) if os.path.basename(cwd) == "notebooks" else cwd
MODEL_DIR = os.path.join(PROJECT_ROOT, "models")
DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
FIG_DIR = os.path.join(PROJECT_ROOT, "docs", "figures")
REPORT_DIR = os.path.join(PROJECT_ROOT, "reports")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Models:", MODEL_DIR)
print("Processed data:", DATA_DIR)

## 2. Load and validate Module 5 and Module 6 outputs

In [ ]:
required_files = {
    "regression_metrics": os.path.join(MODEL_DIR, "regression_metrics.csv"),
    "regression_predictions": os.path.join(DATA_DIR, "regression_test_predictions.csv"),
    "regression_metadata": os.path.join(MODEL_DIR, "regression_model_metadata.json"),
    "multiclass_metrics": os.path.join(MODEL_DIR, "classification_multiclass_metrics.csv"),
    "binary_metrics": os.path.join(MODEL_DIR, "classification_heavy_metrics.csv"),
    "multiclass_predictions": os.path.join(DATA_DIR, "classification_multiclass_test_predictions.csv"),
    "binary_predictions": os.path.join(DATA_DIR, "classification_heavy_test_predictions.csv"),
    "classification_metadata": os.path.join(MODEL_DIR, "classification_model_metadata.json"),
}
missing = [name for name, path in required_files.items() if not os.path.exists(path)]
if missing:
    raise FileNotFoundError("Missing artifacts: " + ", ".join(missing) + ". Run Modules 5 and 6 completely first.")

reg_metrics = pd.read_csv(required_files["regression_metrics"])
reg_pred = pd.read_csv(required_files["regression_predictions"], parse_dates=["date", "target_date"])
multi_metrics = pd.read_csv(required_files["multiclass_metrics"])
binary_metrics = pd.read_csv(required_files["binary_metrics"])
multi_pred = pd.read_csv(required_files["multiclass_predictions"], parse_dates=["date", "target_date"])
binary_pred = pd.read_csv(required_files["binary_predictions"], parse_dates=["date", "target_date"])
with open(required_files["regression_metadata"], encoding="utf-8") as f: reg_meta = json.load(f)
with open(required_files["classification_metadata"], encoding="utf-8") as f: cls_meta = json.load(f)

print("All artifacts loaded.")
print("Regression test rows:", len(reg_pred))
print("Multiclass test rows:", len(multi_pred))
print("Binary test rows:", len(binary_pred))
print("Regression split:", reg_meta.get("split_date"))
print("Classification split:", cls_meta.get("split_date"))
display(reg_metrics.head())
display(multi_metrics.head())
display(binary_metrics.head())

## 3. Confirm that all tasks use the same test period

In [ ]:
def date_span(frame):
    return frame["target_date"].min(), frame["target_date"].max()

spans = {
    "Regression": date_span(reg_pred),
    "Multiclass classification": date_span(multi_pred),
    "Heavy-rain classification": date_span(binary_pred),
}
for name, span in spans.items():
    print(f"{name}: {span[0].date()} to {span[1].date()}")

assert len({(a, b) for a, b in spans.values()}) == 1, "Test date ranges differ. Check that Modules 5 and 6 used the same saved split date."
print("Confirmed: all three tasks use the same chronological test date range.")

## 4. Regression model comparison

In [ ]:
reg_sort = reg_metrics.sort_values(["RMSE", "MAE"], ascending=True).reset_index(drop=True)
display(reg_sort.round(4))
reg_best = reg_sort.iloc[0]
print("Best regression configuration by lowest RMSE:")
print("Model:", reg_best["Model"])
print("Feature set:", reg_best["Feature set"])
print(f"MAE: {reg_best['MAE']:.4f} mm | RMSE: {reg_best['RMSE']:.4f} mm | R²: {reg_best['R2']:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=reg_sort, y="Model", x="RMSE", hue="Feature set", ax=axes[0])
axes[0].set_title("Regression configurations – RMSE (lower is better)")
sns.barplot(data=reg_sort, y="Model", x="MAE", hue="Feature set", ax=axes[1])
axes[1].set_title("Regression configurations – MAE (lower is better)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "m7_01_regression_comparison.png"), dpi=160, bbox_inches="tight")
plt.show()

## 5. Classification model comparison

In [ ]:
print("MULTICLASS – sorted by macro F1")
multi_sort = multi_metrics.sort_values(["macro_f1", "balanced_accuracy"], ascending=False).reset_index(drop=True)
display(multi_sort.round(4))
multi_best = multi_sort.iloc[0]
print("Selected multiclass model:", multi_best["model"], "| Feature set:", multi_best["feature_set"])

print("\nHEAVY-OR-ABOVE BINARY CLASSIFICATION – sorted by heavy-class recall")
binary_sort = binary_metrics.sort_values(["recall_heavy", "f1_heavy", "balanced_accuracy"], ascending=False).reset_index(drop=True)
display(binary_sort.round(4))
binary_best = binary_sort.iloc[0]
print("Selected binary model:", binary_best["model"], "| Feature set:", binary_best["feature_set"])

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
multi_sort["configuration"] = multi_sort["feature_set"] + " | " + multi_sort["model"]
binary_sort["configuration"] = binary_sort["feature_set"] + " | " + binary_sort["model"]
sns.barplot(data=multi_sort, y="configuration", x="macro_f1", ax=axes[0])
axes[0].set_title("Multiclass comparison – macro F1 (higher is better)")
axes[0].set_xlabel("Macro F1")
sns.barplot(data=binary_sort, y="configuration", x="recall_heavy", ax=axes[1])
axes[1].set_title("Heavy-rain comparison – heavy-class recall (higher is better)")
axes[1].set_xlabel("Heavy-class recall")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "m7_02_classification_comparison.png"), dpi=160, bbox_inches="tight")
plt.show()

## 6. Recalculate final test-set metrics from saved predictions

In [ ]:
# Regression metrics from the saved best-model test predictions
y_reg = reg_pred["target_next_mm"].astype(float)
p_reg = reg_pred["predicted_mm"].astype(float)
reg_final = pd.DataFrame([{"task":"Regression – next-day rainfall", "model":reg_meta.get("best_model"),
    "MAE":mean_absolute_error(y_reg,p_reg), "RMSE":np.sqrt(mean_squared_error(y_reg,p_reg)),
    "R2":r2_score(y_reg,p_reg)}])
display(reg_final.round(4))

# Multiclass metrics from saved best-model predictions
multi_target_col = cls_meta["multiclass"]["target"]
actual_multi = multi_pred[multi_target_col].astype(str)
predicted_multi = multi_pred["predicted_class"].astype(str)
multi_final = pd.DataFrame([{"task":"Multiclass rainfall intensity", "model":cls_meta["multiclass"]["best_model"],
    "accuracy":accuracy_score(actual_multi,predicted_multi),
    "macro_precision":precision_score(actual_multi,predicted_multi,average="macro",zero_division=0),
    "macro_recall":recall_score(actual_multi,predicted_multi,average="macro",zero_division=0),
    "macro_f1":f1_score(actual_multi,predicted_multi,average="macro",zero_division=0)}])
display(multi_final.round(4))

# Binary metrics from saved best-model predictions
binary_target_col = cls_meta["heavy_binary"]["target"]
actual_binary = binary_pred[binary_target_col].astype(int)
predicted_binary = binary_pred["predicted_heavy"].astype(int)
binary_final = pd.DataFrame([{"task":"Heavy-or-above rainfall", "model":cls_meta["heavy_binary"]["best_model"],
    "accuracy":accuracy_score(actual_binary,predicted_binary),
    "precision_heavy":precision_score(actual_binary,predicted_binary,zero_division=0),
    "recall_heavy":recall_score(actual_binary,predicted_binary,zero_division=0),
    "f1_heavy":f1_score(actual_binary,predicted_binary,zero_division=0)}])
display(binary_final.round(4))

## 7. Error analysis for rainfall regression

In [ ]:
reg_error = reg_pred[["state","district","date","target_date","target_next_mm","predicted_mm"]].copy()
reg_error["error_mm"] = reg_error["target_next_mm"] - reg_error["predicted_mm"]
reg_error["absolute_error_mm"] = reg_error["error_mm"].abs()
print("Largest absolute errors:")
display(reg_error.nlargest(15, "absolute_error_mm").round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(reg_error["target_next_mm"], reg_error["predicted_mm"], alpha=0.35, s=16)
limit = max(reg_error["target_next_mm"].max(), reg_error["predicted_mm"].max())
axes[0].plot([0, limit], [0, limit], "--", color="black")
axes[0].set(xlabel="Actual rainfall (mm)", ylabel="Predicted rainfall (mm)", title="Actual vs predicted")
sns.histplot(reg_error["error_mm"], bins=40, kde=True, ax=axes[1])
axes[1].axvline(0, linestyle="--", color="black")
axes[1].set_title("Regression residual distribution")
axes[1].set_xlabel("Actual − predicted (mm)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "m7_03_regression_error_analysis.png"), dpi=160, bbox_inches="tight")
plt.show()

## 8. Rainfall intensity and heavy-rain classification diagnostics

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import confusion_matrix, classification_report

labels = cls_meta["multiclass"].get("labels", sorted(actual_multi.unique()))
cm = confusion_matrix(actual_multi, predicted_multi, labels=labels)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=ax)
ax.set(title="Multiclass rainfall intensity – saved test predictions", xlabel="Predicted", ylabel="Actual")
plt.xticks(rotation=35, ha="right"); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "m7_04_multiclass_final_confusion_matrix.png"), dpi=160, bbox_inches="tight"); plt.show()
print(classification_report(actual_multi, predicted_multi, labels=labels, zero_division=0))

cm_bin = confusion_matrix(actual_binary, predicted_binary, labels=[0,1])
fig, ax = plt.subplots(figsize=(6,5))
sns.heatmap(cm_bin, annot=True, fmt="d", cmap="Oranges", xticklabels=["Below heavy", "Heavy or above"], yticklabels=["Below heavy", "Heavy or above"], ax=ax)
ax.set(title="Heavy-rain classification – saved test predictions", xlabel="Predicted", ylabel="Actual")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "m7_05_heavy_final_confusion_matrix.png"), dpi=160, bbox_inches="tight"); plt.show()
print(classification_report(actual_binary, predicted_binary, labels=[0,1], target_names=["Below heavy threshold", "Heavy or above"], zero_division=0))

## 9. Feature importance (when the selected model supports it)

In [ ]:
def load_feature_importance(model_path, feature_names, model_label):
    if not os.path.exists(model_path):
        print("Model file missing:", model_path); return None
    fitted = joblib.load(model_path)
    estimator = fitted.steps[-1][1] if hasattr(fitted, "steps") else fitted
    if hasattr(estimator, "feature_importances_"):
        values = estimator.feature_importances_
    elif hasattr(estimator, "coef_"):
        coef = np.asarray(estimator.coef_)
        values = np.mean(np.abs(coef), axis=0) if coef.ndim > 1 else np.abs(coef)
    else:
        print(model_label, "does not expose native feature importance/coefficients."); return None
    if len(values) != len(feature_names):
        print("Feature count mismatch; skipping importance plot."); return None
    return pd.DataFrame({"feature":feature_names,"importance":values}).sort_values("importance",ascending=False)

reg_meta_path = os.path.join(MODEL_DIR, "regression_model_metadata.json")
reg_model_path = os.path.join(MODEL_DIR, "rainfall_regression_model.pkl")
reg_features = reg_meta.get("features", [])
reg_importance = load_feature_importance(reg_model_path, reg_features, "Regression model")
if reg_importance is not None:
    display(reg_importance.head(15))
    plt.figure(figsize=(9,6)); sns.barplot(data=reg_importance.head(15), y="feature", x="importance")
    plt.title("Regression model – top feature importance / coefficient magnitude")
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,"m7_06_regression_feature_importance.png"),dpi=160,bbox_inches="tight"); plt.show()

cls_model_path = os.path.join(MODEL_DIR, "rainfall_intensity_classifier.pkl")
cls_features = cls_meta["multiclass"].get("features", [])
cls_importance = load_feature_importance(cls_model_path, cls_features, "Multiclass model")
if cls_importance is not None:
    display(cls_importance.head(15))
    plt.figure(figsize=(9,6)); sns.barplot(data=cls_importance.head(15), y="feature", x="importance")
    plt.title("Multiclass model – top feature importance / coefficient magnitude")
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,"m7_07_classification_feature_importance.png"),dpi=160,bbox_inches="tight"); plt.show()

## 10. Final consolidated scorecard

In [ ]:
scorecard = {
    "project": "RainGuard",
    "test_period": {"start": str(spans["Regression"][0].date()), "end": str(spans["Regression"][1].date())},
    "regression": {"model": reg_meta.get("best_model"), "feature_set": reg_meta.get("feature_set"), **{k: float(reg_final.iloc[0][k]) for k in ["MAE","RMSE","R2"]}},
    "multiclass_classification": {"model": cls_meta["multiclass"]["best_model"], "feature_set": cls_meta["multiclass"]["feature_set"], **{k: float(multi_final.iloc[0][k]) for k in ["accuracy","macro_precision","macro_recall","macro_f1"]}},
    "heavy_rain_classification": {"model": cls_meta["heavy_binary"]["best_model"], "feature_set": cls_meta["heavy_binary"]["feature_set"], **{k: float(binary_final.iloc[0][k]) for k in ["accuracy","precision_heavy","recall_heavy","f1_heavy"]}},
    "limitations": ["The dataset covers a short observation window; results do not establish annual or seasonal generalization.", "Heavy-rainfall classification is a rainfall-intensity proxy, not verified flood prediction.", "Performance should be interpreted alongside class-wise recall and the chronological test period."]
}
with open(os.path.join(REPORT_DIR,"rainguard_final_model_scorecard.json"),"w",encoding="utf-8") as f: json.dump(scorecard,f,indent=2)

summary_rows = [
    {"Task":"Regression – next-day rainfall", "Selected model":scorecard["regression"]["model"], "Primary metric":"RMSE (mm)", "Score":scorecard["regression"]["RMSE"]},
    {"Task":"Multiclass rainfall intensity", "Selected model":scorecard["multiclass_classification"]["model"], "Primary metric":"Macro F1", "Score":scorecard["multiclass_classification"]["macro_f1"]},
    {"Task":"Heavy-or-above rainfall", "Selected model":scorecard["heavy_rain_classification"]["model"], "Primary metric":"Heavy recall", "Score":scorecard["heavy_rain_classification"]["recall_heavy"]},
]
summary_df = pd.DataFrame(summary_rows)
display(summary_df.round(4))
summary_df.to_csv(os.path.join(REPORT_DIR,"rainguard_final_model_scorecard.csv"),index=False)
print("Saved final scorecard to:", REPORT_DIR)

## 11. Interpretation and project limitations

Use the actual scorecard and plots above when writing your report. Do not describe a model as “highly accurate” without discussing its test metrics and class-wise errors.

**Important interpretation:**
- Regression estimates next-day rainfall in millimetres. MAE and RMSE are in mm; lower is better. R² describes explained variance and can be negative on a difficult test period.
- Multiclass macro F1 gives equal weight to each rainfall-intensity class, making it useful when classes are imbalanced.
- Heavy-rain recall measures the proportion of actual heavy-or-above cases detected. Higher recall can come with more false alarms, so inspect precision too.
- This dataset and these labels support rainfall forecasting / rainfall-intensity classification. They do **not** establish actual flood occurrence prediction. A genuine flood model needs verified flood-event labels and relevant river, drainage, terrain, soil-moisture and/or water-level information.
- The dataset's short date range limits claims about monsoon seasons, annual patterns and performance in future years.

## 12. Module 7 completion checklist

- [ ] Modules 5 and 6 were run successfully from top to bottom.
- [ ] All saved artifacts loaded without errors.
- [ ] Test date ranges match across regression and classification.
- [ ] Regression, multiclass and heavy-rain results were compared.
- [ ] Error analysis and confusion matrices were reviewed.
- [ ] Final scorecard CSV and JSON were saved in `reports/`.
- [ ] Limitations were included in the project report.